# Week 4, Lab 1 exercise

Second tool for the hand-written tool loop: current weather from wttr.in, bound to the model and called for a city question.

In [ ]:
import os
import requests
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool

load_dotenv(override=True)

llm = ChatOpenAI(model="gpt-5.4-mini")

## Solution

In [ ]:
@tool
def get_weather(city: str) -> str:
    """Return the current weather for a given city using the WETTR API."""
    response = requests.get(f"https://wttr.in/{city}", params={"format": "j1"}, timeout=20)
    response.raise_for_status()
    current = response.json()["current_condition"][0]
    return f"{current['temp_C']}C and {current['weatherDesc'][0]['value']}"

llm_with_tools = llm.bind_tools([get_weather])

In [ ]:
llm_with_weather_tools = llm.bind_tools([get_weather])

# Start the conversation and keep the model's tool request in the history
conversation = [HumanMessage("Wie ist das Wetter in Engen im Hegau?")]
ai_message = llm_with_weather_tools.invoke(conversation)
conversation.append(ai_message)

# Run each requested tool and add its result as a ToolMessage
for call in ai_message.tool_calls:
    if call["name"] == "get_weather":
        result = get_weather.invoke(call["args"])
        conversation.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

# Invoke again, now that the model can see the tool result
final = llm_with_tools.invoke(conversation)
print(final.content)